# S3b — Geometric girth step and training fixes (Colab)

specs/S3b-geometric-girth.md, approved 17 Nov 2026. The notebook only **calls the package**; all logic
is in `src/anthro` (reviewed and tested).

Order matters: every choice is made on **validation people** (held back from training). Test-A and
Test-B are scored **once**, in the last cell, on the configuration you choose.

Before running: **Exécution → Modifier le type d'exécution → GPU T4**.
Cells 3–4 (geometry) only need the CPU, so they also run while GPU time is used up.

## 1. Drive and data

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE = "/content/drive/MyDrive/anthro"
DRIVE_ARCHIVE = "/content/drive/MyDrive/datasets/bodym.tar"
DATA_DIR = "/content/bodym"
os.makedirs(DRIVE, exist_ok=True)

if os.path.exists(DATA_DIR):
    print("Data already on this runtime.")
elif os.path.exists(DRIVE_ARCHIVE):
    !tar -xf "{DRIVE_ARCHIVE}" -C /content
else:
    !pip -q install awscli
    !aws s3 sync --no-sign-request s3://amazon-bodym/ {DATA_DIR} --only-show-errors
    os.makedirs(os.path.dirname(DRIVE_ARCHIVE), exist_ok=True)
    !tar -cf "{DRIVE_ARCHIVE}" -C /content bodym

## 2. Get the code

In [ ]:
BRANCH = "s3b-geometric-girth"   # set to "main" once merged
REPO = "https://github.com/Anthropometry/Anthropometry_Accademic_Model.git"
!rm -rf /content/repo && git clone -q --branch {BRANCH} {REPO} /content/repo
!pip -q install -e /content/repo
!git -C /content/repo log -1 --format="commit %h  %s"
GEO = f"{DRIVE}/geo"          # geometry caches (Drive, never the repo)
CACHE = f"{DRIVE}/cache"      # prepared-image caches

## 3. E0 — measure width and depth on every photo pair (CPU, ~10–20 min the first time)
Full-resolution masks, cached on Drive. **Paste back:** the three E0 tables and any
"could not be measured" lines.

In [ ]:
!anthro geometry --data {DATA_DIR} --out {GEO}

## 4. E1 + E2 — how good is the ellipse, and does the learned correction beat the baseline?
Fitted on training people only (validation people excluded, same split as training).
**Paste back:** everything this cell prints.

In [ ]:
GEO_FIT = f"{GEO}/geo_fit.json"
!anthro geometry-fit --data {DATA_DIR} --cache {GEO} --out {GEO_FIT}

## 5. E3a — fix 1: loss in cm (about 7 min)
**Paste back:** the epoch log and the "validation people at the best epoch" table.

In [ ]:
!anthro train --data {DATA_DIR} --out {DRIVE}/runs/r2 --cache {CACHE} --loss cm

## 6. E3b — fix 1 + fix 2: loss in cm and 512×256 images (about 25 min)
If Colab runs out of GPU memory, change `--batch-size 64` to `32`.
**Paste back:** the epoch log and the validation table.

In [ ]:
!anthro train --data {DATA_DIR} --out {DRIVE}/runs/r3 --cache {CACHE} --loss cm --img-h 512 --img-w 256 --batch-size 64

## 7. E4 — hybrid: the better of E3a/E3b + geometric features
Set `IMG` to the image size that won on **validation** (compare the chest, waist and hip rows of the
two validation tables). **Paste back:** the epoch log and the validation table.

In [ ]:
IMG = "--img-h 256 --img-w 128"      # or "--img-h 512 --img-w 256" if E3b won on validation
!anthro train --data {DATA_DIR} --out {DRIVE}/runs/r4 --cache {CACHE} --loss cm {IMG} --geo-fit {GEO_FIT} --geo-cache {GEO}

## 8. Final — score the chosen run ONCE on Test-A and Test-B (E5 retake warning included)
Set `CHOSEN` to the run with the best validation chest/waist/hip (r2, r3 or r4), decided **before**
running this cell. For the hybrid (r4) the retake warning is chosen on validation people to fire for
10% (D-020). Prints girths and lengths separately, 95% confidence intervals, sex × BMI bands.
**Paste back:** everything this cell prints.

In [ ]:
CHOSEN = "r4"
!anthro evaluate --data {DATA_DIR} --ckpt {DRIVE}/runs/{CHOSEN}/best.pt --cache {CACHE} --geo-cache {GEO} --bootstrap 1000

## Before committing this notebook
**Edit → Clear all outputs**, then save. CI refuses notebooks with saved outputs (POLICY 6).